In [1]:
import sys
sys.path.append('..')
from search.retrieval import *

corpus, inverted_index, bm25_data = load_indices()
faiss_index, faiss_doc_ids, model = load_faiss()
graph, synonyms = load_graph()

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/katharinazeilnhofer/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Loaded: 18114 documents, 76513 tokens


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS loaded: 18114 vectors
Graph loaded: 38694 nodes, 82647 edges


In [2]:
query = "Kartoffelsuppe"

print("=== BM25 ===")
for doc_id, title, score in bm25_search(query, corpus, bm25_data):
    print(f"  [{score:.2f}] {title}")

print("\n=== Semantic Search (FAISS) ===")
for doc_id, title, dist in faiss_search(query, corpus, faiss_index, faiss_doc_ids, model):
    print(f"  [{dist:.4f}] {title}")

print("\n=== Graph Search ===")
for doc_id, title in graph_search(["Kartoffel", "Zwiebel", "Speck"], graph, corpus):
    print(f"  {title}")

=== BM25 ===
  [10.55] Kartoffelsuppe (vegan)
  [10.55] Vegetarische Kartoffelsuppe
  [10.48] Diät Kartoffelsuppe
  [10.48] Kartoffelsuppe (Diät)
  [10.48] Kartoffelsuppe fein

=== Semantic Search (FAISS) ===
  [0.8762] Kartoffelsuppe (vegan)
  [0.8762] Vegetarische Kartoffelsuppe
  [1.0968] Kartoffelsuppen
  [2.7141] Kartoffelsuppe mit Rauchwurst
  [2.9687] Kartoffelsuppe fein

=== Graph Search ===
  Stamppot mit rohen Endivien
  Grünkohl Holsteiner Art
  Klostereintopf mit Rauchfleisch
  Erbsensuppe mit geräucherten Rippen
  Dicke-Bohnen-Stamppot


In [3]:
# without graph filter 
print("=== Hybrid: 'Suppe' ===")
for _, title, score in hybrid_search("Suppe", corpus, bm25_data, faiss_index, faiss_doc_ids, model):
    print(f"  [{score:.4f}] {title}")

# with graph filter
print("\n=== Hybrid: 'Suppe' + Filter 'Kartoffel' ===")
for _, title, score in hybrid_search("Suppe", corpus, bm25_data, faiss_index, faiss_doc_ids, model, 
                                      graph=graph, filter_zutaten=["Kartoffel"]):
    print(f"  [{score:.4f}] {title}")

=== Hybrid: 'Suppe' ===
  [0.8507] Häuptelsalat-Suppe
  [0.8218] Suppe, klar mit Ei
  [0.8218] Suppe, klar mit EI
  [0.8195] Mostsuppe mit Zimtcroûtons
  [0.7988] Rote Bete Suppe mit Meerrettich

=== Hybrid: 'Suppe' + Filter 'Kartoffel' ===
  [0.7613] Ungarische Krautsuppe
  [0.7494] Chinesische Suppe mit Kartoffeln und Fleisch
  [0.7478] Fischsuppe mit Gemüse
  [0.7405] Kartoffelcremesuppe
  [0.7386] Kräutersuppe mit Ei
